# Benchmark do `givp` (Rust) — Funções Clássicas

Notebook de exemplos do **GRASP-ILS-VND-PR** consumido via crate Rust `givp`.
Cada célula escreve um programa Rust completo em um projeto temporário, compila com `cargo` e exibe os resultados.

**Pré-requisito**: Rust toolchain instalado (`rustc`, `cargo`).
O crate local está em `../rust/` — o Cargo.toml de cada exemplo usa `path = "../../rust"`.

## 1. Setup — Helper de Compilação e Execução

In [ ]:
import os
import shutil
import subprocess
import textwrap
from pathlib import Path


# --- Localiza o cargo automaticamente -----------------------------------
def _find_cargo() -> str:
    """Retorna o caminho completo do executável `cargo`, ou lança RuntimeError."""
    found = shutil.which("cargo")
    if found:
        return found
    candidates = [
        Path.home() / ".cargo" / "bin" / "cargo.exe",
        Path.home() / ".cargo" / "bin" / "cargo",
        Path("/usr/local/bin/cargo"),
        Path("/usr/bin/cargo"),
    ]
    for c in candidates:
        if c.is_file():
            return str(c)
    raise RuntimeError(
        "cargo não encontrado. Instale o Rust via https://rustup.rs/ "
        "e reinicie o kernel."
    )


CARGO = _find_cargo()

# --- Caminhos do projeto -------------------------------------------------
PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), "..")))
RUST_CRATE_PATH = PROJECT_ROOT / "rust"
EXAMPLES_DIR = PROJECT_ROOT / "Notebooks" / "_rust_examples"
EXAMPLES_DIR.mkdir(exist_ok=True)


def run_rust_example(name: str, code: str, *, release: bool = False) -> str:
    """Compila e executa um snippet Rust que usa o crate local `givp`."""
    example_dir = EXAMPLES_DIR / name
    src_dir = example_dir / "src"
    src_dir.mkdir(parents=True, exist_ok=True)

    crate_path = os.path.relpath(str(RUST_CRATE_PATH), str(example_dir))
    crate_path_toml = crate_path.replace("\\", "/")

    cargo_toml = textwrap.dedent(
        f"""\
        [package]
        name = "{name}"
        version = "0.1.0"
        edition = "2021"

        [dependencies]
        givp = {{ path = "{crate_path_toml}" }}
    """
    )
    (example_dir / "Cargo.toml").write_text(cargo_toml, encoding="utf-8")
    (src_dir / "main.rs").write_text(code, encoding="utf-8")

    cmd = [CARGO, "run", "--quiet"]
    if release:
        cmd.append("--release")

    result = subprocess.run(  # noqa: S603
        cmd, cwd=str(example_dir), capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"[ERRO]\n{result.stderr}"
    return result.stdout.strip()


cargo_version = subprocess.run(  # noqa: S603
    [CARGO, "--version"], capture_output=True, text=True
).stdout.strip()
print(f"Projeto raiz : {PROJECT_ROOT}")
print(f"Crate givp   : {RUST_CRATE_PATH}")
print(f"Exemplos em  : {EXAMPLES_DIR}")
print(f"Rust         : {cargo_version}")
print("Setup concluído.")

Projeto raiz : d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr
Crate givp   : d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr\rust
Exemplos em  : d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr\Notebooks\_rust_examples
Rust         : cargo 1.95.0 (f2d3ce0bd 2026-03-21)
Setup concluído.


## 2. Problema 1: Sphere

$$f(x) = \sum_{i=1}^{n} x_i^2, \quad x \in [-5.12, 5.12]^{10}, \quad f^* = 0$$

```rust
use givp::{givp, GivpConfig};

fn main() {
    let func = |x: &[f64]| x.iter().map(|v| v * v).sum::<f64>();
    let bounds = vec![(-5.12_f64, 5.12_f64); 10];
    let config = GivpConfig { seed: Some(42), max_iterations: 60, ..GivpConfig::default() };
    let res = givp(func, &bounds, config).unwrap();
    println!("best_fun = {:.6e}  nfev = {}", res.fun, res.nfev);
}
```

In [2]:
sphere_code = """\
use givp::{givp, GivpConfig};

fn main() {
    let func = |x: &[f64]| -> f64 { x.iter().map(|v| v * v).sum() };
    let bounds: Vec<(f64, f64)> = vec![(-5.12, 5.12); 10];
    let config = GivpConfig {
        seed: Some(42),
        max_iterations: 60,
        alpha: 0.15,
        vnd_iterations: 30,
        ils_iterations: 8,
        early_stop_threshold: 40,
        ..GivpConfig::default()
    };
    let res = givp(func, &bounds, config).unwrap();
    println!("best_fun = {:.6e}", res.fun);
    println!("nfev     = {}", res.nfev);
    println!("success  = {}", res.success);
    println!("message  = {}", res.message);
    let x_str: Vec<String> = res.x.iter().map(|v| format!("{v:.4}")).collect();
    println!("x        = [{}]", x_str.join(", "));
}
"""

print("Compilando e executando Sphere...")
output = run_rust_example("sphere", sphere_code)
print(output)

Compilando e executando Sphere...
best_fun = 8.325563e-5
nfev     = 202038
success  = true
message  = early stop due to stagnation
x        = [0.0014, 0.0021, 0.0011, -0.0000, 0.0022, 0.0009, -0.0010, 0.0037, 0.0003, -0.0074]


## 3. Problema 2: Rosenbrock

$$f(x) = \sum_{i=1}^{n-1} \left[100(x_{i+1} - x_i^2)^2 + (1-x_i)^2\right], \quad x^* = (1,\dots,1),\ f^* = 0$$

```rust
fn rosenbrock(x: &[f64]) -> f64 {
    x.windows(2).map(|w| 100.0 * (w[1] - w[0].powi(2)).powi(2) + (1.0 - w[0]).powi(2)).sum()
}
```

In [3]:
rosenbrock_code = """\
use givp::{givp, GivpConfig};

fn rosenbrock(x: &[f64]) -> f64 {
    x.windows(2)
        .map(|w| 100.0 * (w[1] - w[0].powi(2)).powi(2) + (1.0 - w[0]).powi(2))
        .sum()
}

fn main() {
    let bounds: Vec<(f64, f64)> = vec![(-2.048, 2.048); 5];
    let config = GivpConfig {
        seed: Some(42),
        max_iterations: 60,
        alpha: 0.15,
        vnd_iterations: 30,
        ils_iterations: 8,
        early_stop_threshold: 40,
        ..GivpConfig::default()
    };
    let res = givp(rosenbrock, &bounds, config).unwrap();
    println!("best_fun = {:.6e}", res.fun);
    println!("nfev     = {}", res.nfev);
    let x_str: Vec<String> = res.x.iter().map(|v| format!("{v:.4}")).collect();
    println!("x        = [{}]", x_str.join(", "));
    println!("x* err   = {:.4e}", res.x.iter().map(|v| (v - 1.0).powi(2)).sum::<f64>().sqrt());
}
"""

print("Compilando e executando Rosenbrock...")
output = run_rust_example("rosenbrock", rosenbrock_code)
print(output)

Compilando e executando Rosenbrock...
best_fun = 1.533693e-2
nfev     = 132485
x        = [0.9955, 1.0015, 1.0070, 1.0133, 1.0319]
x* err   = 3.5632e-2


## 4. Problema 3: Rastrigin (multimodal)

$$f(x) = 10n + \sum_{i=1}^{n} \left[x_i^2 - 10\cos(2\pi x_i)\right], \quad x^* = 0,\ f^* = 0$$

Altamente multimodal — testa capacidade de escapar de muitos ótimos locais.

In [4]:
rastrigin_code = """\
use givp::{givp, GivpConfig};
use std::f64::consts::PI;

fn rastrigin(x: &[f64]) -> f64 {
    let n = x.len() as f64;
    10.0 * n + x.iter().map(|&xi| xi * xi - 10.0 * (2.0 * PI * xi).cos()).sum::<f64>()
}

fn main() {
    let bounds: Vec<(f64, f64)> = vec![(-5.12, 5.12); 10];
    let config = GivpConfig {
        seed: Some(42),
        max_iterations: 80,
        alpha: 0.12,
        vnd_iterations: 50,
        ils_iterations: 10,
        early_stop_threshold: 60,
        ..GivpConfig::default()
    };
    let res = givp(rastrigin, &bounds, config).unwrap();
    println!("best_fun = {:.6e}", res.fun);
    println!("nfev     = {}", res.nfev);
    let x_str: Vec<String> = res.x.iter().map(|v| format!("{v:.4}")).collect();
    println!("x        = [{}]", x_str.join(", "));
}
"""

print("Compilando e executando Rastrigin...")
output = run_rust_example("rastrigin", rastrigin_code)
print(output)

Compilando e executando Rastrigin...
best_fun = 2.098117e-1
nfev     = 635928
x        = [0.0017, -0.0254, -0.0143, 0.0015, -0.0038, -0.0044, 0.0017, -0.0063, 0.0066, 0.0093]


## 5. Problema 4: Ackley

$$f(x) = -20\exp\!\left(-0.2\sqrt{\tfrac{1}{n}\sum x_i^2}\right)
         -\exp\!\left(\tfrac{1}{n}\sum\cos(2\pi x_i)\right) + 20 + e$$

$x \in [-32.768, 32.768]^{10}$, $f^* = 0$

In [5]:
ackley_code = """\
use givp::{givp, GivpConfig};
use std::f64::consts::{E, PI};

fn ackley(x: &[f64]) -> f64 {
    let n = x.len() as f64;
    let sum_sq: f64 = x.iter().map(|&xi| xi * xi).sum();
    let sum_cos: f64 = x.iter().map(|&xi| (2.0 * PI * xi).cos()).sum();
    -20.0 * (-0.2 * (sum_sq / n).sqrt()).exp()
        - (sum_cos / n).exp()
        + 20.0
        + E
}

fn main() {
    let bounds: Vec<(f64, f64)> = vec![(-32.768, 32.768); 10];
    let config = GivpConfig {
        seed: Some(42),
        max_iterations: 80,
        ..GivpConfig::default()
    };
    let res = givp(ackley, &bounds, config).unwrap();
    println!("best_fun = {:.6e}", res.fun);
    println!("nfev     = {}", res.nfev);
}
"""

print("Compilando e executando Ackley...")
output = run_rust_example("ackley", ackley_code)
print(output)

Compilando e executando Ackley...
best_fun = 1.176768e-1
nfev     = 694437


## 6. Maximização: Griewank Negativo

Demonstra o uso de `Direction::Maximize` passando o negativo da Griewank.
O `givp` maximiza quando `direction = Direction::Maximize` — internamente o sinal é invertido.

```rust
let config = GivpConfig {
    direction: Direction::Maximize,
    ..GivpConfig::default()
};
```

In [6]:
griewank_max_code = """\
use givp::{givp, Direction, GivpConfig};
use std::f64::consts::SQRT_2;

/// Griewank negada — maximizar este valor equivale a minimizar Griewank.
fn neg_griewank(x: &[f64]) -> f64 {
    let sum_sq: f64 = x.iter().map(|&xi| xi * xi / 4000.0).sum();
    let prod_cos: f64 = x.iter()
        .enumerate()
        .map(|(i, &xi)| (xi / (i as f64 + 1.0).sqrt()).cos())
        .product();
    -(sum_sq - prod_cos + 1.0)
}

fn main() {
    let bounds: Vec<(f64, f64)> = vec![(-600.0, 600.0); 10];
    let config = GivpConfig {
        seed: Some(42),
        max_iterations: 80,
        direction: Direction::Maximize,
        ..GivpConfig::default()
    };
    let res = givp(neg_griewank, &bounds, config).unwrap();
    // res.fun é o valor de neg_griewank no ótimo (negativo de Griewank)
    println!("neg_griewank(x*) = {:.6e}  (Griewank ≈ {:.6e})", res.fun, -res.fun);
    println!("nfev     = {}", res.nfev);
    println!("direction = {:?}", res.termination);
}
"""

print("Compilando e executando maximização (Griewank negado)...")
output = run_rust_example("griewank_max", griewank_max_code)
print(output)

Compilando e executando maximização (Griewank negado)...
neg_griewank(x*) = -1.521676e-1  (Griewank â‰ˆ 1.521676e-1)
nfev     = 722544
direction = MaxIterationsReached


## 7. Problema Misto: Knapsack 0/1

Variáveis contínuas em $[0,1]$ — itens selecionados pelo limiar $> 0.5$.
O campo `integer_split` do `GivpConfig` pode ser utilizado quando as primeiras
variáveis são inteiras e as restantes contínuas; aqui usamos a codificação
contínua com penalização.

In [7]:
knapsack_code = """\
use givp::{givp, GivpConfig};

const N: usize = 15;
const CAPACITY: f64 = 100.0;
const VALUES:  [f64; N] = [55.0, 45.0, 90.0, 70.0, 40.0, 30.0, 80.0, 60.0, 50.0, 35.0, 20.0, 75.0, 65.0, 25.0, 85.0];
const WEIGHTS: [f64; N] = [20.0, 15.0, 40.0, 30.0, 10.0,  5.0, 35.0, 25.0, 18.0, 12.0,  8.0, 28.0, 22.0,  6.0, 38.0];

fn knapsack_penalty(x: &[f64]) -> f64 {
    let mut total_value = 0.0_f64;
    let mut total_weight = 0.0_f64;
    for (i, &xi) in x.iter().enumerate() {
        if xi > 0.5 {
            total_value  += VALUES[i];
            total_weight += WEIGHTS[i];
        }
    }
    let overflow = (total_weight - CAPACITY).max(0.0);
    -total_value + 1000.0 * overflow
}

fn main() {
    let bounds: Vec<(f64, f64)> = vec![(0.0, 1.0); N];
    let config = GivpConfig {
        seed: Some(42),
        max_iterations: 80,
        ..GivpConfig::default()
    };
    let res = givp(knapsack_penalty, &bounds, config).unwrap();
    let selected: Vec<usize> = res.x.iter().enumerate()
        .filter_map(|(i, &v)| if v > 0.5 { Some(i) } else { None })
        .collect();
    let total_val: f64 = selected.iter().map(|&i| VALUES[i]).sum();
    let total_w: f64   = selected.iter().map(|&i| WEIGHTS[i]).sum();
    println!("itens selecionados = {:?}", selected);
    println!("valor total        = {:.0}", total_val);
    println!("peso total         = {:.0}  (cap={})", total_w, CAPACITY);
    println!("best_fun           = {:.2}", res.fun);
}
"""

print("Compilando e executando Knapsack 0/1...")
output = run_rust_example("knapsack", knapsack_code)
print(output)

Compilando e executando Knapsack 0/1...
itens selecionados = [0, 4, 5, 10, 11, 12, 13]
valor total        = 310
peso total         = 99  (cap=100)
best_fun           = -310.00


## 8. Reproducibilidade com `seed`

Com `seed: Some(42)`, duas chamadas idênticas devem retornar exatamente o mesmo resultado.

In [8]:
seed_code = """\
use givp::{givp, GivpConfig};

fn sphere(x: &[f64]) -> f64 { x.iter().map(|v| v * v).sum() }

fn main() {
    let bounds: Vec<(f64, f64)> = vec![(-5.12, 5.12); 5];
    let cfg = || GivpConfig { seed: Some(42), max_iterations: 30, ..GivpConfig::default() };

    let r1 = givp(sphere, &bounds, cfg()).unwrap();
    let r2 = givp(sphere, &bounds, cfg()).unwrap();

    println!("run1: fun={:.10e}  nfev={}", r1.fun, r1.nfev);
    println!("run2: fun={:.10e}  nfev={}", r2.fun, r2.nfev);
    println!("reproducível: {}", (r1.fun - r2.fun).abs() < 1e-12);
}
"""

print("Compilando e executando teste de reproducibilidade...")
output = run_rust_example("seed_test", seed_code)
print(output)

Compilando e executando teste de reproducibilidade...
run1: fun=5.3407826909e-5  nfev=72768
run2: fun=5.3407826909e-5  nfev=72768
reproducÃ­vel: true


## 9. Limpeza dos artefatos de compilação (opcional)

Remova a pasta temporária `_rust_examples/` quando não precisar mais dos binários compilados.

In [9]:
# Descomente para remover os artefatos de compilação
# import shutil
# if EXAMPLES_DIR.exists():
#     shutil.rmtree(EXAMPLES_DIR)
#     print(f"Removido: {EXAMPLES_DIR}")
# else:
#     print("Nada para remover.")

print(f"Exemplos compilados em: {EXAMPLES_DIR}")
print("Para limpar, descomente o bloco acima.")

Exemplos compilados em: d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr\Notebooks\_rust_examples
Para limpar, descomente o bloco acima.
